# <span style="color: #1F1DB5;">SPRINT 10 – Evaluación y Selección de Modelos

# <span style="color: #1F1DB5;">Notebook 2 – Comparación de Modelos y Cross-Validation — Versión estudiante

## Cómo usar esta versión estudiante

Este notebook está preparado para que tomes apuntes, completes ejercicios y documentes tus decisiones durante la clase.

**Modo de trabajo recomendado:**

1. Lee primero la consigna de cada bloque o ejercicio.
2. Completa los espacios de respuesta antes de comparar con la explicación del instructor/a.
3. Ejecuta las celdas de setup cuando existan.
4. Escribe interpretación, dudas y decisiones en los espacios de apuntes.
5. Al final, revisa si tus respuestas conectan datos, método e implicación de negocio.

> Las salidas de ejecución fueron limpiadas para que puedas correr el notebook desde cero.


## Notas de clase principales

- Entender por qué una sola partición train/test no es suficiente para evaluar un modelo.
- Implementar k-Fold Cross-Validation y su variante estratificada.
- Usar crossvalscore para obtener estimaciones robustas de desempeño.
- Comparar Decision Tree vs Random Forest vs Logistic Regression de forma sistemática.
- Aplicar GridSearchCV para búsqueda de hiperparámetros.

### Mis notas iniciales

- 
- 
- 


## <span style="color: #2826DE;">Objetivos de Aprendizaje

- Entender por qué una **sola partición train/test** no es suficiente para evaluar un modelo.
- Implementar **k-Fold Cross-Validation** y su variante estratificada.
- Usar `cross_val_score` para obtener estimaciones robustas de desempeño.
- Comparar **Decision Tree vs Random Forest vs Logistic Regression** de forma sistemática.
- Aplicar **GridSearchCV** para búsqueda de hiperparámetros.
- Comprender el **tradeoff sesgo-varianza** y su relación con learning curves.

### <span style="color: #2772F2;">Agenda (2 horas)

| Tema | Duración |
|---|---|
¿Por qué una sola partición no basta? | 10 min |
k-Fold Cross-Validation | 15 min |
cross_val_score: Evaluación robusta | 10 min |
Comparación sistemática de modelos | 20 min |
GridSearchCV: Búsqueda de hiperparámetros | 15 min |
Bias-Variance tradeoff y learning curves | 15 min |
Actividad: Pair Programming | 15 min |
Tips y buenas prácticas | 5 min |
Cierre y Kahoot | 5 min |

## <span style="color: #2826DE;">❓ Pregunta Guía

### Si entrenas un modelo 10 veces con diferentes particiones de datos y obtienes accuracies entre 0.65 y 0.95, ¿cuál es el "verdadero" desempeño del modelo? ¿En cuál de esos números confiarías?

### Mi respuesta inicial

- 


## <span style="color: #2826DE;">¿Por qué una sola partición no basta? (10 mins)

Cuando hacemos un solo train/test split, nuestro resultado depende fuertemente de **qué datos cayeron en cada conjunto**. Es como evaluar a un chef por un solo platillo — tal vez tuvo un buen día o los ingredientes estaban perfectos.

**El problema concreto:**
- Con `random_state=42` obtienes accuracy = 0.87
- Con `random_state=0` obtienes accuracy = 0.79
- Con `random_state=123` obtienes accuracy = 0.92

¿Cuál es el verdadero desempeño? Ninguno por sí solo es confiable.

**Cross-Validation** resuelve esto evaluando el modelo en MÚLTIPLES particiones diferentes y promediando los resultados. Es como evaluar al chef con 10 platillos distintos — obtenemos una imagen mucho más confiable de su verdadera habilidad.

Demostremos la variabilidad entre diferentes splits aleatorios:

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Dataset Wine: clasificación multiclase
wine = load_wine()
X = pd.DataFrame(wine.data, columns=wine.feature_names)
y = pd.Series(wine.target)

# Evaluamos el MISMO modelo con 20 splits diferentes
accuracies = []
for seed in range(20):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=seed)
    dt = DecisionTreeClassifier(random_state=42)
    dt.fit(X_tr, y_tr)
    acc = accuracy_score(y_te, dt.predict(X_te))
    accuracies.append(acc)

print(f"Accuracy con 20 splits diferentes (mismo modelo):")
print(f"  Mínimo:  {min(accuracies):.4f}")
print(f"  Máximo:  {max(accuracies):.4f}")
print(f"  Media:   {np.mean(accuracies):.4f}")
print(f"  Std:     {np.std(accuracies):.4f}")
print(f"  Rango:   {max(accuracies) - min(accuracies):.4f}")

plt.figure(figsize=(10, 4))
plt.bar(range(20), accuracies, color='steelblue', alpha=0.7)
plt.axhline(np.mean(accuracies), color='red', linestyle='--', label=f'Media: {np.mean(accuracies):.3f}')
plt.xlabel('Random seed')
plt.ylabel('Accuracy')
plt.title('Variabilidad del accuracy según la partición elegida')
plt.legend()
plt.tight_layout()
plt.show()

Preguntas:

- ¿Es justo comparar dos modelos si cada uno fue evaluado con un split diferente?

- ¿Cuánta variabilidad consideras "aceptable" entre particiones?

- ¿Cómo afecta el tamaño del dataset a esta variabilidad?

### Mis respuestas de validación

- 
- 


## <span style="color: #2826DE;">k-Fold Cross-Validation (15 mins)

**k-Fold CV** divide el dataset en **k partes iguales** (folds). Luego entrena el modelo k veces, cada vez usando un fold diferente como test y los restantes k-1 como train.

**El proceso para k=5:**
1. Divide datos en 5 folds: [F1, F2, F3, F4, F5]
2. Iteración 1: Train=[F2,F3,F4,F5], Test=[F1] → score₁
3. Iteración 2: Train=[F1,F3,F4,F5], Test=[F2] → score₂
4. Iteración 3: Train=[F1,F2,F4,F5], Test=[F3] → score₃
5. Iteración 4: Train=[F1,F2,F3,F5], Test=[F4] → score₄
6. Iteración 5: Train=[F1,F2,F3,F4], Test=[F5] → score₅
7. Resultado final: promedio ± desviación estándar

**Ventajas:**
- Cada dato se usa tanto para entrenar como para evaluar (pero nunca al mismo tiempo).
- Obtenemos media y desviación estándar — sabemos qué tan estable es el modelo.
- Aprovechamos mejor los datos (importante con datasets pequeños).

**Stratified k-Fold:** Variante que mantiene la proporción de clases en cada fold. **Siempre** usarla en clasificación con clases desbalanceadas.

Implementemos k-Fold CV paso a paso para entender el mecanismo, y luego con el shortcut de sklearn:

In [ ]:
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score

# Implementación manual para entender el mecanismo
kf = KFold(n_splits=5, shuffle=True, random_state=42)

print("=== k-Fold CV Manual (k=5) ===\n")
scores_manual = []
for i, (train_idx, test_idx) in enumerate(kf.split(X)):
    X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
    y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
    
    dt = DecisionTreeClassifier(random_state=42)
    dt.fit(X_tr, y_tr)
    score = dt.score(X_te, y_te)
    scores_manual.append(score)
    print(f"  Fold {i+1}: train={len(train_idx)}, test={len(test_idx)}, accuracy={score:.4f}")

print(f"\n  Promedio: {np.mean(scores_manual):.4f} ± {np.std(scores_manual):.4f}")

# Shortcut con cross_val_score (¡una sola línea!)
print("\n=== cross_val_score (mismo resultado, una línea) ===\n")
dt = DecisionTreeClassifier(random_state=42)
scores_cv = cross_val_score(dt, X, y, cv=5, scoring='accuracy')
print(f"  Scores: {scores_cv.round(4)}")
print(f"  Promedio: {scores_cv.mean():.4f} ± {scores_cv.std():.4f}")

# Stratified k-Fold (mantiene proporciones)
print("\n=== Stratified k-Fold (recomendado para clasificación) ===\n")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores_strat = cross_val_score(dt, X, y, cv=skf, scoring='accuracy')
print(f"  Scores: {scores_strat.round(4)}")
print(f"  Promedio: {scores_strat.mean():.4f} ± {scores_strat.std():.4f}")

Preguntas:

- ¿Qué valor de k (folds) elegirías para un dataset con 50 muestras? ¿Y para uno con 50,000?

- ¿Por qué Stratified k-Fold es especialmente importante en datasets desbalanceados?

- ¿Cross-validation reemplaza al test set final? ¿O necesitas ambos?

### Mis respuestas de validación

- 
- 


## <span style="color: #2826DE;">Comparación sistemática de modelos (20 mins)

Ahora que sabemos evaluar modelos de forma robusta, comparemos tres algoritmos populares de forma justa y sistemática:

1. **Decision Tree:** Modelo interpretable que divide el espacio con reglas if/else. Propenso a overfitting.
2. **Random Forest:** Ensemble de muchos árboles. Más robusto pero menos interpretable.
3. **Logistic Regression:** Modelo lineal clásico. Simple, rápido, funciona bien con relaciones lineales.

**Principios para una comparación justa:**
- Usar exactamente los **mismos folds** de CV para todos los modelos.
- Comparar la **misma métrica** para todos.
- Reportar media **Y** desviación estándar (un modelo con score 0.85±0.01 es mejor que uno con 0.86±0.10).
- Considerar la **complejidad** del modelo (a igual desempeño, prefiere el más simple).

Comparemos los tres modelos usando cross-validation con múltiples métricas:

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_validate

# Definimos los modelos (LR necesita escalado, lo metemos en un pipeline)
models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('lr', LogisticRegression(random_state=42, max_iter=1000))
    ])
}

# Evaluamos con múltiples métricas
scoring = ['accuracy', 'f1_macro', 'recall_macro']
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = {}
print("=" * 70)
print(f"{'Modelo':<25} {'Accuracy':<18} {'F1 Macro':<18} {'Recall Macro':<18}")
print("=" * 70)

for name, model in models.items():
    cv_results = cross_validate(model, X, y, cv=cv, scoring=scoring)
    results[name] = cv_results
    
    acc = cv_results['test_accuracy']
    f1 = cv_results['test_f1_macro']
    rec = cv_results['test_recall_macro']
    
    print(f"{name:<25} {acc.mean():.3f}±{acc.std():.3f}    "
          f"{f1.mean():.3f}±{f1.std():.3f}    "
          f"{rec.mean():.3f}±{rec.std():.3f}")

print("=" * 70)

Visualicemos la distribución de scores para entender mejor la estabilidad de cada modelo:

In [ ]:
# Visualización de la comparación
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
metrics = ['test_accuracy', 'test_f1_macro', 'test_recall_macro']
titles = ['Accuracy', 'F1 Macro', 'Recall Macro']

for ax, metric, title in zip(axes, metrics, titles):
    data = [results[name][metric] for name in models.keys()]
    bp = ax.boxplot(data, labels=list(models.keys()), patch_artist=True)
    
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Score')
    ax.tick_params(axis='x', rotation=15)
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Comparación de modelos con Cross-Validation (5-Fold)', fontweight='bold')
plt.tight_layout()
plt.show()

# Resumen final
print("\n📊 Resumen de la comparación:")
print("-" * 50)
best_model = max(results.keys(), key=lambda x: results[x]['test_f1_macro'].mean())
print(f"  Mejor modelo por F1 macro: {best_model}")
print(f"  Score: {results[best_model]['test_f1_macro'].mean():.4f} ± "
      f"{results[best_model]['test_f1_macro'].std():.4f}")

Preguntas:

- ¿Por qué Random Forest suele superar a Decision Tree?

- Si Logistic Regression tiene un score similar a Random Forest, ¿cuál elegirías en producción y por qué?

- ¿Cuándo un modelo con mayor variabilidad (std alta) podría ser preocupante?

### Mis respuestas de validación

- 
- 


## <span style="color: #2826DE;">GridSearchCV: Búsqueda de hiperparámetros (15 mins)

Los hiperparámetros son las "perillas" que ajustamos ANTES de entrenar el modelo. Ejemplos:
- Decision Tree: `max_depth`, `min_samples_split`
- Random Forest: `n_estimators`, `max_depth`, `max_features`
- KNN: `n_neighbors`, `metric`

**GridSearchCV** automatiza la búsqueda:
1. Define un "grid" de combinaciones a probar.
2. Para CADA combinación, hace cross-validation completa.
3. Devuelve la mejor combinación y su score.

**Advertencia:** El grid crece exponencialmente. Si tienes 3 parámetros con 5 valores cada uno = 125 combinaciones × 5 folds = 625 entrenamientos. ¡Elige tus rangos con cuidado!

Usemos GridSearchCV para optimizar Random Forest sobre el dataset Wine:

In [ ]:
from sklearn.model_selection import GridSearchCV

# Definimos el grid de hiperparámetros a explorar
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 10, None],
    'min_samples_split': [2, 5, 10]
}

# GridSearchCV: prueba TODAS las combinaciones con CV
rf = RandomForestClassifier(random_state=42)
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='f1_macro',
    n_jobs=-1,       # Usar todos los cores
    verbose=0,
    return_train_score=True
)

grid_search.fit(X, y)

print("=== Resultados de GridSearchCV ===\n")
print(f"Total de combinaciones evaluadas: {len(grid_search.cv_results_['mean_test_score'])}")
print(f"\nMejores hiperparámetros:")
for param, value in grid_search.best_params_.items():
    print(f"  {param}: {value}")
print(f"\nMejor F1 macro (CV): {grid_search.best_score_:.4f}")

# Top 5 combinaciones
results_df = pd.DataFrame(grid_search.cv_results_)
top5 = results_df.nlargest(5, 'mean_test_score')[
    ['param_n_estimators', 'param_max_depth', 'param_min_samples_split',
     'mean_test_score', 'std_test_score', 'mean_train_score']
].reset_index(drop=True)
top5.columns = ['n_estimators', 'max_depth', 'min_samples_split', 
                'test_score', 'test_std', 'train_score']
print(f"\nTop 5 combinaciones:")
print(top5.to_string(index=False))

Preguntas:

- ¿Qué pasa si la diferencia entre train_score y test_score es muy grande?

- ¿Por qué es importante usar CV dentro de GridSearch y no solo un split?

- ¿Cuáles son las alternativas a GridSearch cuando el espacio de búsqueda es enorme?

### Mis respuestas de validación

- 
- 


## <span style="color: #2826DE;">Bias-Variance tradeoff y learning curves (15 mins)

El tradeoff sesgo-varianza es quizás el concepto más importante en Machine Learning:

- **Sesgo alto (underfitting):** El modelo es demasiado simple. No captura los patrones. Error alto en TRAIN y TEST.
- **Varianza alta (overfitting):** El modelo es demasiado complejo. Memoriza el ruido. Error bajo en TRAIN, alto en TEST.

**Learning curves** nos permiten diagnosticar si nuestro modelo sufre de high bias o high variance:

| Diagnóstico | Train score | Test score | Solución |
|---|---|---|---|
| **High bias** | Bajo | Bajo (similar a train) | Modelo más complejo, más features |
| **High variance** | Alto | Bajo (gap grande) | Más datos, regularización, modelo más simple |
| **Equilibrio** | Alto | Alto (cercano a train) | ¡Buen modelo! |

Grafiquemos learning curves para nuestros tres modelos y diagnostiquemos su comportamiento:

In [ ]:
from sklearn.model_selection import learning_curve

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

models_lc = {
    'Decision Tree\n(max_depth=None)': DecisionTreeClassifier(random_state=42),
    'Random Forest\n(n_estimators=100)': RandomForestClassifier(n_estimators=100, random_state=42),
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('lr', LogisticRegression(random_state=42, max_iter=1000))
    ])
}

for ax, (name, model) in zip(axes, models_lc.items()):
    train_sizes, train_scores, test_scores = learning_curve(
        model, X, y, cv=5, n_jobs=-1, 
        train_sizes=np.linspace(0.1, 1.0, 10),
        scoring='accuracy', random_state=42
    )
    
    train_mean = train_scores.mean(axis=1)
    train_std = train_scores.std(axis=1)
    test_mean = test_scores.mean(axis=1)
    test_std = test_scores.std(axis=1)
    
    ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1, color='blue')
    ax.fill_between(train_sizes, test_mean - test_std, test_mean + test_std, alpha=0.1, color='orange')
    ax.plot(train_sizes, train_mean, 'o-', color='blue', label='Train')
    ax.plot(train_sizes, test_mean, 'o-', color='orange', label='Validation')
    
    ax.set_xlabel('Tamaño del training set')
    ax.set_ylabel('Accuracy')
    ax.set_title(name, fontweight='bold')
    ax.legend(loc='lower right')
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0.5, 1.05)
    
    # Diagnóstico
    gap = train_mean[-1] - test_mean[-1]
    if gap > 0.1:
        ax.annotate('⚠️ High Variance', xy=(0.5, 0.05), xycoords='axes fraction',
                   fontsize=9, color='red', fontweight='bold')
    elif test_mean[-1] < 0.8:
        ax.annotate('⚠️ High Bias', xy=(0.5, 0.05), xycoords='axes fraction',
                   fontsize=9, color='red', fontweight='bold')

plt.suptitle('Learning Curves: Diagnóstico de Bias vs Variance', fontweight='bold')
plt.tight_layout()
plt.show()

print("\n📋 Interpretación:")
print("  - Si train ≈ 1.0 y test << train → High variance (overfitting)")
print("  - Si train ≈ test pero ambos bajos → High bias (underfitting)")
print("  - Si train ≈ test y ambos altos → Buen equilibrio")

Preguntas:

- Mirando las learning curves, ¿cuál modelo muestra más overfitting?

- Si obtienes más datos, ¿a cuál de los tres modelos le beneficiaría más?

- ¿Qué técnicas conoces para reducir la varianza de un Decision Tree?

### Mis respuestas de validación

- 
- 


## <span style="color: #2826DE;">Actividad: Pair Programming (15 mins)

### Espacio de trabajo del estudiante

**Respuesta, decisiones o interpretación:**

- 
- 


Trabajarán en parejas: uno escribe código (driver) y el otro guía la lógica (navigator). Cambien roles a la mitad del tiempo.

### Instrucciones:

**Objetivo:** Comparar 3 modelos con CV y optimizar el mejor con GridSearchCV.

**Driver (escribe código) + Navigator (guía la lógica). Cambien roles a los 7 minutos.**

1. Carga el dataset Wine (`load_wine()`).
2. Define 3 modelos: DecisionTree, RandomForest, LogisticRegression (con Pipeline+Scaler).
3. Usa `cross_val_score` con StratifiedKFold(n_splits=5) y `scoring='f1_macro'`.
4. Imprime media ± std de cada modelo. Identifica el mejor.
5. Para el mejor modelo, define un `param_grid` con al menos 2 hiperparámetros.
6. Ejecuta GridSearchCV y reporta los mejores hiperparámetros y score.

**Bonus:** Grafica las learning curves del modelo antes y después de la optimización.

**Tiempo:** 12 minutos de trabajo + 3 minutos de discusión grupal.

## <span style="color: #2826DE;">Tips y buenas prácticas

- **Siempre usa Stratified k-Fold** para problemas de clasificación.
- Reporta siempre **media ± desviación estándar**, nunca solo la media.
- Empieza con modelos simples (Logistic Regression) como **baseline** antes de probar modelos complejos.
- Usa `Pipeline` para encapsular preprocesamiento + modelo — evita data leakage en CV.
- GridSearchCV es exhaustivo pero lento. Para espacios grandes, considera `RandomizedSearchCV`.
- Learning curves son tu mejor herramienta de diagnóstico — úsalas siempre que un modelo no funcione bien.
- El modelo más complejo NO siempre es el mejor. A igual desempeño, elige el más simple (Occam's Razor).

## <span style="color: #2826DE;">Cierre y Kahoot (5 mins)

Kahoot - Preguntas sugeridas

1️⃣ ¿Cuántas veces se entrena un modelo en 5-Fold Cross-Validation?

- 1 vez
- 5 veces 
- 10 veces
- Depende del dataset


2️⃣ ¿Qué ventaja tiene Stratified k-Fold sobre k-Fold regular?

- Es más rápido de calcular
- Mantiene la proporción de clases en cada fold 
- Usa menos memoria RAM
- Elimina outliers automáticamente


3️⃣ Si train_score=0.99 y test_score=0.65, tu modelo probablemente tiene:

- High bias (underfitting)
- High variance (overfitting) 
- Buen equilibrio
- Un bug en el código


4️⃣ GridSearchCV combina:

- Solo búsqueda de hiperparámetros
- Solo cross-validation
- Búsqueda de hiperparámetros + cross-validation 
- Entrenamiento con todo el dataset sin validación


5️⃣ ¿Cuál es una desventaja de GridSearchCV?

- No funciona con Random Forest
- El costo computacional crece exponencialmente con el número de parámetros 
- Solo puede optimizar un hiperparámetro a la vez
- Siempre produce overfitting


6️⃣ Si ambos scores (train y test) son bajos y similares, el modelo tiene:

- High variance
- High bias 
- Data leakage
- Desempeño perfecto